In [34]:
# STW7082CEM: Big Data Management and Data Visualisation
# Student: Sandesh Tamang [cite: 4]
# Implementation: Real-Time Misinformation and Fake News Analysis [cite: 1]

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. Initialize Spark Session (Big Data Management [cite: 16])
spark = SparkSession.builder \
    .appName("FakeNewsPropagationAnalysis") \
    .config("spark.sql.debug.maxToStringFields", "100") \
    .getOrCreate()

# 2. Data Ingestion (Task requirement: 500+ rows [cite: 128])
# We use multiLine=True and quote/escape handling to prevent CSV shifting errors
df = spark.read.csv("/dataset/Features_For_Traditional_ML_Techniques.csv", 
                     header=True, 
                     inferSchema=True, 
                     multiLine=True, 
                     quote='"', 
                     escape='"')

# 3. Robust Data Cleaning (Addressing the NumberFormatException)
# We cast to double; malformed text entries will safely turn into NULL [cite: 20]
numeric_cols = [
    'BinaryNumTarget', 'followers_count', 'friends_count', 'retweets', 
    'favourites', 'mentions', 'quotes', 'replies', 'total_count', 
    'Word count', 'Average word length'
]

cleaned_df = df
for c in numeric_cols:
    cleaned_df = cleaned_df.withColumn(c, col(c).cast("double"))

# Filter out rows where the label or crucial metrics are NULL due to corruption
# This ensures we have a valid dataset for analysis [cite: 135]
final_df = cleaned_df.filter(
    col("BinaryNumTarget").isNotNull() & 
    col("retweets").isNotNull()
).withColumnRenamed("BinaryNumTarget", "label")

# 4. Propagation Analysis (Objective: Compare Fake vs Real [cite: 21, 45])
print("Summary of News Propagation Signatures:")
propagation_stats = final_df.groupBy("label").agg(
    avg("retweets").alias("Avg_Retweets"),
    avg("followers_count").alias("Avg_Reach"),
    avg("total_count").alias("Avg_Engagement_Volume")
)
propagation_stats.show()

# 5. Feature Engineering & Classification (Task requirement: PySpark ML )
# Selecting attributes to satisfy the 10+ attribute requirement [cite: 127, 128]
feature_columns = [
    'followers_count', 'friends_count', 'retweets', 'favourites', 
    'mentions', 'quotes', 'replies', 'total_count', 
    'Word count', 'Average word length'
]

# Assemble features into a vector for Spark ML
assembler = VectorAssembler(inputCols=feature_columns, outputCol="raw_features", handleInvalid="skip")
assembled_data = assembler.transform(final_df)

# Scaling features for better Logistic Regression performance
scaler = StandardScaler(inputCol="raw_features", outputCol="features")
scaler_model = scaler.fit(assembled_data)
scaled_data = scaler_model.transform(assembled_data)

# Split data: 70% Training, 30% Testing
train_data, test_data = scaled_data.randomSplit([0.7, 0.3], seed=42)

# Train Logistic Regression Model 
lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train_data)

# 6. Evaluation
predictions = model.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol="label")
auc = evaluator.evaluate(predictions)
print(f"Model Performance (Area Under ROC): {auc}")

# 7. Data Export for Tableau (Task requirement: Tableau exclusively [cite: 85])
# Exporting the cleaned aggregations for the interactive dashboard [cite: 39]
propagation_stats.toPandas().to_csv("tableau_results.csv", index=False)
print("Data exported successfully for Tableau.")

Summary of News Propagation Signatures:


26/03/08 20:08:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , majority_target, statement, BinaryNumTarget, tweet, followers_count, friends_count, favourites_count, statuses_count, listed_count, following, embeddings, BotScore, BotScoreBinary, cred, normalize_influence, mentions, quotes, replies, retweets, favourites, hashtags, URLs, unique_count, total_count, ORG_percentage, NORP_percentage, GPE_percentage, PERSON_percentage, MONEY_percentage, DATE_percentage, CARDINAL_percentage, PERCENT_percentage, ORDINAL_percentage, FAC_percentage, LAW_percentage, PRODUCT_percentage, EVENT_percentage, TIME_percentage, LOC_percentage, WORK_OF_ART_percentage, QUANTITY_percentage, LANGUAGE_percentage, Word count, Max word length, Min word length, Average word length, present_verbs, past_verbs, adjectives, adverbs, adpositions, pronouns, TOs, determiners, conjunctions, dots, exclamation, questions, ampersand, capitals, digits, long_word_freq, short_word_freq
 Schema: _c

+-----+-----------------+-----------------+---------------------+
|label|     Avg_Retweets|        Avg_Reach|Avg_Engagement_Volume|
+-----+-----------------+-----------------+---------------------+
|  0.0|7.857617821903536|5128.313308206165|    3.339017589017589|
|  1.0| 5.55395328594226|17130.33917017264|   3.5380095749310896|
+-----+-----------------+-----------------+---------------------+



26/03/08 20:08:55 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , majority_target, statement, BinaryNumTarget, tweet, followers_count, friends_count, favourites_count, statuses_count, listed_count, following, embeddings, BotScore, BotScoreBinary, cred, normalize_influence, mentions, quotes, replies, retweets, favourites, hashtags, URLs, unique_count, total_count, ORG_percentage, NORP_percentage, GPE_percentage, PERSON_percentage, MONEY_percentage, DATE_percentage, CARDINAL_percentage, PERCENT_percentage, ORDINAL_percentage, FAC_percentage, LAW_percentage, PRODUCT_percentage, EVENT_percentage, TIME_percentage, LOC_percentage, WORK_OF_ART_percentage, QUANTITY_percentage, LANGUAGE_percentage, Word count, Max word length, Min word length, Average word length, present_verbs, past_verbs, adjectives, adverbs, adpositions, pronouns, TOs, determiners, conjunctions, dots, exclamation, questions, ampersand, capitals, digits, long_word_freq, short_word_freq
 Schema: _c

Model Performance (Area Under ROC): 0.579182904038112


26/03/08 20:09:03 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , majority_target, statement, BinaryNumTarget, tweet, followers_count, friends_count, favourites_count, statuses_count, listed_count, following, embeddings, BotScore, BotScoreBinary, cred, normalize_influence, mentions, quotes, replies, retweets, favourites, hashtags, URLs, unique_count, total_count, ORG_percentage, NORP_percentage, GPE_percentage, PERSON_percentage, MONEY_percentage, DATE_percentage, CARDINAL_percentage, PERCENT_percentage, ORDINAL_percentage, FAC_percentage, LAW_percentage, PRODUCT_percentage, EVENT_percentage, TIME_percentage, LOC_percentage, WORK_OF_ART_percentage, QUANTITY_percentage, LANGUAGE_percentage, Word count, Max word length, Min word length, Average word length, present_verbs, past_verbs, adjectives, adverbs, adpositions, pronouns, TOs, determiners, conjunctions, dots, exclamation, questions, ampersand, capitals, digits, long_word_freq, short_word_freq
 Schema: _c

Data exported successfully for Tableau.


26/03/09 00:01:36 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 935061 ms exceeds timeout 120000 ms
26/03/09 00:01:36 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/09 00:02:05 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at o